In [ ]:
suppressWarnings({
    library(data.table)
    library(geosphere)
    source("preprocess_utils.R")
    library(parallel)
    library(ggplot2)
    library(feather)
    library(dplyr)
    library(tidyverse)
})

landing_ac_data <- ac_data <- read_feather("full_ac_data.feather")
print(nrow(ac_data))
setDT(ac_data)

In [ ]:
ground_speed_limit <- 3000 # knots
max_climb_rate <- 200 # ft/s
min_reqd_alt <- 10000 # ft
max_ground_speed_est_ratio <- 20
max_sample_gap_sec <- 1800
setorder(landing_ac_data, id, ts)

In [ ]:
next_ts_ms <- suppressWarnings(as.numeric(as.character(ac_data$next_ts)))
ts_ms <- suppressWarnings(as.numeric(as.character(ac_data$ts)))
ac_data$time_diff_condition <- is.na(next_ts_ms) |
  (next_ts_ms - ts_ms) > max_sample_gap_sec * 1000
ac_data$alt_diff_condition <- ((ac_data$altitude - ac_data$last_altitude) < 0) |
                                ((ac_data$next_altitude - ac_data$altitude) < 0)


ac_data$landing_condition <- data.table::between(x = ac_data$lon, lower = -75, upper = -72.5) &
                      data.table::between(x = ac_data$lat, lower = 40, upper = 42) &
                        ac_data$altitude <= 1500 & ac_data$alt_diff_condition &
                        ac_data$time_diff_condition
ac_data[, "landing" := cumsum(landing_condition), by=id]
ac_data[, "land_max" := max(landing), by = id]
landing_ac_data <- ac_data[ac_data$land_max != 0, ]

In [ ]:
length(unique(landing_ac_data$id))

In [ ]:
# Keeping first row of lon, lat for each aircraft
setorder(landing_ac_data, id, ts)
landing_ac_data[, c("last_lon", "last_lat") := list(c(NA, lon[-.N]), c(NA, lat[-.N])), by = "id"]
landing_ac_data <- landing_ac_data[landing_ac_data$lon != landing_ac_data$last_lon | landing_ac_data$lat != landing_ac_data$last_lat, ]
print(nrow(ac_data))
landing_ac_data[, c("last_ts", "last_lon", "last_lat") := list(c(NA, ts[-.N]), c(NA, lon[-.N]), c(NA, lat[-.N])), by = "id"]

In [ ]:
landing_ac_summary <- landing_ac_data %>% group_by(id) %>% summarize(max_landings = max(land_max, na.rm = T))
hist(landing_ac_summary$max_landings)

In [ ]:
nrow(landing_ac_data)
landing_ac_data <- landing_ac_data[, .SD[-c(1)], by = id]
nrow(landing_ac_data)
landing_ac_data[, "landing_ac" := data.table::between(x = lon[.N], lower = -74.2, upper = -73.3) &
                                          data.table::between(x = lat[.N], lower = 40.3, upper = 41) &
                altitude[.N] <= 1500,
                                            by = id]
landing_ac_data <- landing_ac_data[landing_ac_data$landing_ac, ]
nrow(landing_ac_data)

In [ ]:
landing_ac_data[, "alt" := altitude[.N], by = id]
range(landing_ac_data$alt)

In [ ]:
landing_ac_data[, c("max_alt") := list(max(altitude, na.rm = T)), by = id]
landing_ac_data <- landing_ac_data[landing_ac_data$max_alt >= min_reqd_alt, ]
nrow(landing_ac_data)

In [ ]:
distance_m <- apply(landing_ac_data[, c("lon", "lat", "last_lon", "last_lat")], 1, ground_dist)
delta_ms <- suppressWarnings(as.numeric(as.character(landing_ac_data$ts))) -
  suppressWarnings(as.numeric(as.character(landing_ac_data$last_ts)))
landing_ac_data$ground_speed_est <- ground_speed_knots(distance_m, delta_ms)
landing_ac_data[, max_ground_speed_est :=
  suppressWarnings(max(ground_speed_est, na.rm = TRUE)), by = id]
landing_ac_data[!is.finite(max_ground_speed_est), max_ground_speed_est := NA_real_]


In [ ]:
mean(landing_ac_data$max_ground_speed_est <= ground_speed_limit)

In [ ]:
hist(landing_ac_data$ground_speed_est/landing_ac_data$ground_speed, breaks = 1000)

In [ ]:
mean(landing_ac_data$ground_speed_est/landing_ac_data$ground_speed <= 20)
landing_ac_data[, "max_ratio" := max(ground_speed_est/ground_speed), by = id]
landing_ac_data1 <- unique(landing_ac_data[, c("id", "max_ratio")])
hist(landing_ac_data1$max_ratio)

In [ ]:
nrow(landing_ac_data)
mean(landing_ac_data1$max_ratio <= max_ground_speed_est_ratio)
reqd_ids <- landing_ac_data1$id[landing_ac_data1$max_ratio <= max_ground_speed_est_ratio]
landing_ac_data <- landing_ac_data[sapply(landing_ac_data$id, function(id1) id1 %in% reqd_ids), ]
nrow(landing_ac_data)

In [ ]:
landing_ac_data <- landing_ac_data[
  is.finite(max_ground_speed_est) & max_ground_speed_est <= ground_speed_limit
]
print(nrow(landing_ac_data))


In [ ]:
landing_ac_data[, "last_alt" := c(NA, altitude[-.N]), by = id]
ht_df <- landing_ac_data[landing_ac_data$altitude != landing_ac_data$last_alt, ]
setorder(ht_df, id, ts)

In [ ]:
nrow(ht_df)

In [ ]:
ht_df[, "last_ts" := c(NA, ts[-.N]), by = id]
delta_sec <- (suppressWarnings(as.numeric(as.character(ht_df$ts))) -
              suppressWarnings(as.numeric(as.character(ht_df$last_ts)))) / 1000
ht_df$climb_rate <- ifelse(
  is.finite(delta_sec) & delta_sec > 0,
  (ht_df$altitude - ht_df$last_alt) / delta_sec,
  NA_real_
) # ft/s

In [ ]:
ht_df[, max_abs_climb_rate :=
  suppressWarnings(max(abs(climb_rate), na.rm = TRUE)), by = id]
ht_df[!is.finite(max_abs_climb_rate), max_abs_climb_rate := NA_real_]
hist(ht_df$max_abs_climb_rate, breaks = 100)

In [ ]:
mean(ht_df$max_abs_climb_rate <= max_climb_rate)
clean_landing_id <- unique(ht_df$id[
  is.finite(ht_df$max_abs_climb_rate) & ht_df$max_abs_climb_rate <= max_climb_rate
])
length(clean_landing_id)

In [ ]:
landing_ac_data <- landing_ac_data[sapply(landing_ac_data$id, function(id1) id1 %in% clean_landing_id), ]

In [ ]:
nrow(landing_ac_data)

landing_ac_data <- filter_m_min_r_rows(landing_ac_data)

In [ ]:
length(unique(landing_ac_data$id))

In [ ]:
world <- map_data("world")
a_region <- filter(world, long > -78 & long < -68 & lat > 35 & lat < 45)

In [ ]:
dir.create("clean_landed_id", showWarnings = F)
setwd("clean_landed_id/")
uniq_id <- unique(landing_ac_data$aircraft)
plot_ids <- plot_aircraft_ids(jfk_data1 = landing_ac_data, a_region = a_region,
                              min_lon = min(landing_ac_data$lon), max_lon = max(landing_ac_data$lon),
                              min_lat = min(landing_ac_data$lat), max_lat = max(landing_ac_data$lat))
suppressWarnings({
    plot_ids(uniq_id = uniq_id)
})
# landing_ac_data <- get_quadrant(landing_ac_data)

In [ ]:
write_feather(landing_ac_data, "landing_subset_refined.feather")